### Подготовка данных и предобработка

В этом разделе мы загружаем датасет с результатами кластеризации пациентов.

In [ ]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

df = pd.read_csv('patient_segmentation_dataset_with_clusters.xls')

X = df.drop(['PatientID', 'Last_Visit_Date', 'cluster_kmeans', 'cluster_dbscan', 'cluster_agg'], axis=1)
y = df['cluster_kmeans']

X = X.fillna('Unknown')
for col in X.select_dtypes(include=['object']).columns:
    X[col] = LabelEncoder().fit_transform(X[col])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Обучение и оценка моделей

Мы сравниваем три различных алгоритма классификации, чтобы определить, какой из них лучше всего предсказывает принадлежность пациента к кластеру.

In [ ]:
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(random_state=42),
    "SVC": SVC()
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    results[name] = accuracy_score(y_test, y_pred)
    print(f"\nModel: {name}\n", classification_report(y_test, y_pred))

Судя по отчетам классификации Logistic Regression показала наилучший результат с точностью 98%.

In [ ]:
best_model_name = max(results, key=results.get)
joblib.dump(models[best_model_name], 'classification_model.kpl')
print(f"Saved best model: {best_model_name}")